<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/07_cross_dataset_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Notebook 07 — Cross-dataset performance analysis
# Uses only results produced by Notebooks 01 and 06.
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_ROOT = Path("/content/drive2/MyDrive/fraud-detection-data")
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_ROOT}. Mount Google Drive at /content/drive2 first."
    )

# All outputs are written directly to fraud-detection-data; no new folders.
COMPARISON_DIR = DATA_ROOT

required_files = [
    DATA_ROOT / "ieee_test_results.csv",
    DATA_ROOT / "paysim_results.csv",
    DATA_ROOT / "stacking_experiment_summary.json",
    DATA_ROOT / "notebook01_run_manifest.json",
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required Notebook 01/05/06 outputs are missing:\\n" + "\\n".join(missing)
    )

ieee_df = pd.read_csv(DATA_ROOT / "ieee_test_results.csv")
paysim_df = pd.read_csv(DATA_ROOT / "paysim_results.csv")
with open(DATA_ROOT / "stacking_experiment_summary.json", encoding="utf-8") as f:
    stacking_summary = json.load(f)
with open(DATA_ROOT / "notebook01_run_manifest.json", encoding="utf-8") as f:
    data_manifest = json.load(f)

print("IEEE-CIS results:", ieee_df.shape)
print("PaySim results:", paysim_df.shape)
print("\\nIEEE-CIS test results:")
display(ieee_df.round(5))
print("\\nPaySim test results:")
display(paysim_df.round(5))


In [ ]:
# Dataset characteristics from Notebook 01's actual-data manifest.
# IEEE-CIS figures refer to its labelled training file; PaySim figures refer
# to the full PaySim source file. They are descriptive, not test-set metrics.
dataset_comparison = pd.DataFrame([
    {
        "Dataset": "IEEE-CIS",
        "Source_Rows": int(data_manifest["ieee_train_rows"]),
        "Fraud_Cases": int(data_manifest["ieee_train_fraud_count"]),
        "Fraud_Rate_Percent": float(data_manifest["ieee_train_fraud_rate_percent"]),
        "Count_Basis": "Labelled IEEE-CIS training file"
    },
    {
        "Dataset": "PaySim",
        "Source_Rows": int(data_manifest["paysim_rows"]),
        "Fraud_Cases": int(data_manifest["paysim_fraud_count"]),
        "Fraud_Rate_Percent": float(data_manifest["paysim_fraud_rate_percent"]),
        "Count_Basis": "Full PaySim source file"
    }
])

dataset_comparison.to_csv(
    COMPARISON_DIR / "cross_dataset_dataset_comparison.csv",
    index=False
)
display(dataset_comparison)


In [ ]:
#Identify the best model for each dataset
ieee_best = ieee_df.sort_values(
    "PR_AUC", ascending=False
).iloc[0]

paysim_best = paysim_df.sort_values(
    "PR_AUC", ascending=False
).iloc[0]

best_models = pd.DataFrame([
    {
        "Dataset": "IEEE-CIS",
        "Best_Model": ieee_best["Model"],
        "Precision": ieee_best["Precision"],
        "Recall": ieee_best["Recall"],
        "F1": ieee_best["F1"],
        "ROC_AUC": ieee_best["ROC_AUC"],
        "PR_AUC": ieee_best["PR_AUC"]
    },
    {
        "Dataset": "PaySim",
        "Best_Model": (
            paysim_best["Feature_Set"] + " " +
            paysim_best["Model"]
        ),
        "Precision": paysim_best["Precision"],
        "Recall": paysim_best["Recall"],
        "F1": paysim_best["F1"],
        "ROC_AUC": paysim_best["ROC_AUC"],
        "PR_AUC": paysim_best["PR_AUC"]
    }
])

best_models.to_csv(
    COMPARISON_DIR / "cross_dataset_best_model_comparison.csv",
    index=False
)

display(best_models.round(4))

In [ ]:
#Plot IEEE-CIS model performance
plt.figure(figsize=(10, 5))

plt.bar(
    ieee_df["Model"],
    ieee_df["PR_AUC"]
)

plt.title("IEEE-CIS Model Comparison — PR-AUC")
plt.xlabel("Model")
plt.ylabel("PR-AUC")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    COMPARISON_DIR / "cross_dataset_ieee_model_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
#Plot PaySim model performance
plt.figure(figsize=(9, 5))

paysim_labels = (
    paysim_df["Feature_Set"] + " " +
    paysim_df["Model"]
)

plt.bar(
    paysim_labels,
    paysim_df["PR_AUC"]
)

plt.title("PaySim Feature Set and Model Comparison — PR-AUC")
plt.xlabel("Feature Set and Model")
plt.ylabel("PR-AUC")
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    COMPARISON_DIR / "cross_dataset_paysim_model_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
# Summarize results without treating scores across different datasets as directly comparable.
research_conclusion = f"""CROSS-DATASET MODEL RESULTS

IEEE-CIS best by test PR-AUC: {ieee_best['Model']} (PR-AUC {ieee_best['PR_AUC']:.4f}).
PaySim best by test PR-AUC: {paysim_best['Feature_Set']} {paysim_best['Model']} (PR-AUC {paysim_best['PR_AUC']:.4f}).

The datasets have different feature spaces, fraud prevalence, time periods, and validation/test protocols. Compare models within each dataset; do not interpret the raw PR-AUC values as a cross-dataset ranking.

PaySim's extended feature set includes post-transaction balances. Its results are an offline comparison and should not be presented as real-time performance. Use the RealTime feature set for the deployable scenario.

Thresholds are selected using validation data. Report test results once after model and threshold selection.
"""
with open(COMPARISON_DIR / "cross_dataset_research_conclusion.txt", "w") as f:
    f.write(research_conclusion)
print(research_conclusion)


In [ ]:
#Combine all experiment results
ieee_export = ieee_df.copy()
ieee_export["Dataset"] = "IEEE-CIS"

paysim_export = paysim_df.copy()
paysim_export["Dataset"] = "PaySim"

combined_results = pd.concat(
    [ieee_export, paysim_export],
    ignore_index=True,
    sort=False
)

combined_results.to_csv(
    COMPARISON_DIR / "cross_dataset_all_experiment_results.csv",
    index=False
)

print("All cross-dataset analysis outputs saved directly in fraud-detection-data.")
print("\nFiles created:")

for filename in sorted(DATA_ROOT.iterdir()):
    print("-", filename.name)

print("\\nNotebook 07 generated files:")
for path in sorted(DATA_ROOT.glob("cross_dataset_*")):
    if path.is_file():
        print(f"{path.name} — {path.stat().st_size / (1024**2):.2f} MB")
